In [1]:
# agentic_workflow.py

# TODO: 1 - Import the following agents: ActionPlanningAgent, KnowledgeAugmentedPromptAgent, EvaluationAgent, RoutingAgent from the workflow_agents.base_agents module
from workflow_agents.base_agents import ActionPlanningAgent, KnowledgeAugmentedPromptAgent, EvaluationAgent, RoutingAgent
import os
from dotenv import load_dotenv

# TODO: 2 - Load the OpenAI key into a variable called openai_api_key
load_dotenv()
openai_api_key=os.getenv("OPENAI_API_KEY")

# load the product spec
# TODO: 3 - Load the product spec document Product-Spec-Email-Router.txt into a variable called product_spec
product_spec=open("Product-Spec-Email-Router.txt").read()

In [2]:
# Instantiate all the agents
# Action Planning Agent
knowledge_action_planning = (
    "Stories are defined from a product spec by identifying a persona, an action, and a desired outcome for each story. \n"
    "Each story represents a specific functionality of the product described in the specification. \n"
    "Features are defined by grouping related user stories. \n"
    "Tasks are defined for each story and represent the engineering work required to develop the product. \n"
    "A development Plan for a product contains all these components"
)
# TODO: 4 - Instantiate an action_planning_agent using the 'knowledge_action_planning'
action_planning_agent=ActionPlanningAgent(openai_api_key, knowledge_action_planning)

In [3]:
# Product Manager - Knowledge Augmented Prompt Agent
persona_product_manager = "You are a Product Manager, you are responsible for defining the user stories for a product."
knowledge_product_manager = (
    "Stories are defined by writing sentences with a persona, an action, and a desired outcome. "
    "The sentences always start with: As a "
    "Write several stories for the product spec below, where the personas are the different users of the product. "
    # TODO: 5 - Complete this knowledge string by appending the product_spec loaded in TODO 3
    + product_spec
)
product_manager_knowledge_agent=KnowledgeAugmentedPromptAgent(openai_api_key,persona_product_manager,knowledge_product_manager)

In [4]:
persona_product_manager_eval = "You are Product Manager evaluation agent. You assess responses from a Product Manager worker agent."
product_manager_evaluation_criteria="""Evaluate responses against these criteria:
**User Story Structure**: Must follow the format:
   'As a [type of user], I want [an action or feature] so that [benefit/value].'
   - [type of user] must be a specific persona, not generic (e.g., 'admin user' not just 'user')
   - [action or feature] must be concrete and measurable
   - [benefit/value] must articulate clear business or user value"""

product_manager_evaluation_agent = EvaluationAgent(
    openai_api_key, 
    persona_product_manager_eval,
    product_manager_evaluation_criteria, 
    product_manager_knowledge_agent, 
    20
    )

In [5]:
persona_program_manager = "You are a Program Manager, you are responsible for defining the features for a product."
knowledge_program_manager = "Features of a product are defined by organizing similar user stories into cohesive groups."

# Instantiate a program_manager_knowledge_agent using 'persona_program_manager' and 'knowledge_program_manager'
# (This is a necessary step before TODO 8. Students should add the instantiation code here.)
program_manager_knowledge_agent=KnowledgeAugmentedPromptAgent(openai_api_key,persona_program_manager,knowledge_program_manager)

# Program Manager - Evaluation Agent
persona_program_manager_eval = "You are an evaluation agent that checks the answers of other worker agents."

program_manager_evaluation_criteria = """The answer should be product features that follow the following structure: 
                      Feature Name: A clear, concise title that identifies the capability
                      Description: A brief explanation of what the feature does and its purpose
                      Key Functionality: The specific capabilities or actions the feature provides
                      User Benefit: How this feature creates value for the user"""

program_manager_evaluation_agent = EvaluationAgent(
    openai_api_key, 
    persona_program_manager_eval,
    program_manager_evaluation_criteria, 
    program_manager_knowledge_agent, 
    20
    )

In [15]:
# Development Engineer - Knowledge Augmented Prompt Agent
persona_dev_engineer = "You are a Development Engineer, you are responsible for defining the development tasks for a product."
knowledge_dev_engineer = """Development tasks are defined by identifying what needs to be built to implement each user story.
You MUST output actual tasks, not a description of how to create tasks. 
Each task must contain real content, not placeholders or process descriptions.
"""
# Instantiate a development_engineer_knowledge_agent using 'persona_dev_engineer' and 'knowledge_dev_engineer'
# (This is a necessary step before TODO 9. Students should add the instantiation code here.)
development_engineer_knowledge_agent=KnowledgeAugmentedPromptAgent(openai_api_key,persona_dev_engineer,knowledge_dev_engineer)
# Development Engineer - Evaluation Agent
persona_dev_engineer_eval = "You are an evaluation agent that checks the answers of other worker agents"
dev_engineer_evaluation_criteria="""The answer should contain ACTUAL development tasks (not a description of how to create them).
Each task MUST follow this exact structure with real, specific content:
    Task ID: A unique identifier for tracking purposes
    Task Title: Brief description of the specific development work
    Related User Story: Reference to the parent user story
    Description: Detailed explanation of the technical work required
    Acceptance Criteria: Specific requirements that must be met for completion
    Estimated Effort: Time or complexity estimation
    Dependencies: Any tasks that must be completed first"""

dev_engineer_evaluation_agent = EvaluationAgent(
    openai_api_key, 
    persona_dev_engineer_eval,
    dev_engineer_evaluation_criteria, 
    development_engineer_knowledge_agent, 
    20
    )

In [7]:
# Run the workflow

print("\n*** Workflow execution started ***\n")
# Workflow Prompt
# ****
workflow_prompt = "What would the development tasks for this product be?"
# ****
print(f"Task to complete in this workflow, workflow prompt = {workflow_prompt}")

print("\nDefining workflow steps from the workflow prompt")
# TODO: 12 - Implement the workflow.
#   1. Use the 'action_planning_agent' to extract steps from the 'workflow_prompt'.
#   2. Initialize an empty list to store 'completed_steps'.
#   3. Loop through the extracted workflow steps:
#      a. For each step, use the 'routing_agent' to route the step to the appropriate support function.
#      b. Append the result to 'completed_steps'.
#      c. Print information about the step being executed and its result.
#   4. After the loop, print the final output of the workflow (the last completed step).


steps=action_planning_agent.extract_steps_from_prompt(workflow_prompt)
print(steps)


*** Workflow execution started ***

Task to complete in this workflow, workflow prompt = What would the development tasks for this product be?

Defining workflow steps from the workflow prompt
['1. Review the user stories defined for the product.', '2. Break down each user story into specific tasks required for implementation.', '3. Estimate the time and resources needed for each task.', '4. Prioritize the tasks based on dependencies and critical path.', '5. Assign tasks to team members or developers.', '6. Set deadlines for each task to track progress.', '7. Monitor the progress of tasks and make adjustments as needed.', '8. Conduct regular meetings to discuss task status and address any issues.', '9. Test the completed tasks to ensure they meet the acceptance criteria defined in the user stories.', '10. Iterate on tasks based on feedback and testing results.']


In [16]:
def product_manager_support_function(input_query):
    response=product_manager_knowledge_agent.respond(input_query)
    evaluation=product_manager_evaluation_agent.evaluate(response)
    return evaluation["final_response"]

def program_manager_support_function(input_query):
    response=program_manager_knowledge_agent.respond(input_query)
    evaluation=program_manager_evaluation_agent.evaluate(response)
    return evaluation["final_response"]

def dev_engineer_support_function(input_query):
    response=development_engineer_knowledge_agent.respond(input_query)
    evaluation=dev_engineer_evaluation_agent.evaluate(response)
    return evaluation["final_response"]

agents = [
    {
        "name": "Product Manager agent",
        "description": "Resolves tasks related to a Product Manager",
        "func": lambda x: product_manager_support_function(x) 
    },
    {
        "name": "Program Manager agent",
        "description": "Resolves tasks related to a Program Manager",
        "func": lambda x: program_manager_support_function(x) 
    },
    {
        "name": "Development Engineer agent",
        "description": "Resolves tasks related to a Development Engineer",
        "func": lambda x: dev_engineer_support_function(x)
    }
]
routing_agent = RoutingAgent(openai_api_key, agents)

In [9]:
steps[0]

'1. Review the user stories defined for the product.'

In [12]:
result=routing_agent.route(steps[0])

[Router] Best agent: Product Manager agent (score=0.352)
✅ Final solution accepted after 1 steps
Yes, the response meets the criteria for User Story Structure. Each user persona is clearly defined (Customer Support Representative, Subject Matter Expert, IT Administrator), the action or feature they want is specific and measurable (automate responses, intelligently route inquiries, provide a comprehensive dashboard), and the benefit/value is clearly articulated for each persona (focus on complex issues efficiently, provide specialized assistance effectively, ensure smooth operation and make necessary adjustments easily).
As a Customer Support Representative, I want the Email Router system to automate responses to routine inquiries so that I can focus on addressing complex customer issues efficiently. This will reduce the time spent on repetitive tasks and allow me to provide better support to customers with more complex issues.

As a Subject Matter Expert (SME), I want the Email Router 

In [11]:
print(result)

As a Customer Support Representative, I want the Email Router system to automate responses to routine inquiries so that I can focus on addressing complex customer issues efficiently. 

As a Subject Matter Expert (SME), I want the Email Router system to intelligently route complex inquiries to me based on content analysis so that I can provide specialized assistance effectively. 

As an IT Administrator, I want the Email Router system to provide a comprehensive dashboard for monitoring system performance metrics so that I can ensure the system is running smoothly and efficiently. 

As a User, I want the Email Router system to allow manual intervention to override automated processes when necessary so that I can ensure accurate and appropriate responses are provided to customers.


In [9]:
steps[1]

'2. Break down each user story into specific tasks required for implementation.'

In [17]:
result1=routing_agent.route(steps[1])

[Router] Best agent: Development Engineer agent (score=0.343)
✅ Final solution accepted after 2 steps
Yes, the answer meets the criteria as it provides a list of tasks following the specified structure with Task ID, Task Title, Related User Story, Description, Acceptance Criteria, Estimated Effort, and Dependencies for each task.
1. Task Identifier: DEV-001
   Related User Stories: US-001
   Description: Research and analyze the user story requirements to understand the scope and objectives.
   Acceptance Criteria: Documented summary of user story requirements and key features identified.
   Estimated Effort: 2 days
   Dependencies: None

2. Task Identifier: DEV-002
   Related User Stories: US-001
   Description: Define and create the data model structure required to support the user story functionality.
   Acceptance Criteria: Data model schema designed and documented.
   Estimated Effort: 3 days
   Dependencies: DEV-001

3. Task Identifier: DEV-003
   Related User Stories: US-001
   

In [14]:
result1

"As a Customer Support Representative, I want the Email Router system to automate responses to routine inquiries so that I can focus on resolving complex customer issues effectively and efficiently.\n\nAs a Subject Matter Expert (SME), I want the Email Router system to intelligently route complex inquiries to me based on content analysis so that I can provide specialized expertise where it is most needed and improve my efficiency.\n\nAs an IT Administrator, I want the Email Router system to provide a comprehensive dashboard for monitoring system performance metrics so that I can easily configure, maintain, and ensure optimal performance of the system within the organization's existing IT infrastructure."

In [ ]:
completed_steps=[]

for i, step in enumerate(steps):
    print(f"\n--- Step {i+1}: {step} ---")
    result=routing_agent.route(step)
    completed_steps.append(result)
    print(f"Result: {result}")

print(f"\n*** Final Output ***\n{completed_steps}")